# CatBoost Model Optimisation

This notebook keeps the same final comparison design used for the other tuned models:

1. Model settings are selected without using the threshold-validation or final-test sets.
2. The probability threshold is selected only on the validation set.
3. The final test set is evaluated once after the model settings and threshold are fixed.
4. The validation recall target is 80%.

The main methodological difference is that CatBoost receives the original categorical variables directly. They are **not one-hot encoded before training**. This allows CatBoost to use its own categorical-feature statistics and categorical combinations.

Like the XGBoost notebook, an internal early-stopping split is used to choose the learning rate and number of boosting rounds. After those choices are fixed, the final CatBoost model is refitted on the complete 64% model-training set.

In [ ]:
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import sklearn
import catboost

from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)
from sklearn.model_selection import (
    ParameterSampler,
    StratifiedKFold,
    train_test_split
)

print("scikit-learn version:", sklearn.__version__)
print("CatBoost version:", catboost.__version__)

scikit-learn version: 1.9.0
CatBoost version: 1.2.10


## 1. Load the cleaned dataset and create the output folder

In [2]:
PROJECT_ROOT = Path.cwd()

for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    candidate = (
        parent
        / "Processed_Dataset"
        / "diabetic_data_cleaned_stage1.csv"
    )

    if candidate.exists():
        DATA_PATH = candidate
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError(
        "Could not find "
        "Processed_Dataset/diabetic_data_cleaned_stage1.csv"
    )

OUTPUT_DIR = (
    PROJECT_ROOT
    / "Model_Results"
    / "catboost_optimisation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

df = pd.read_csv(DATA_PATH)

print("Dataset path:")
print(DATA_PATH)

print("\nDataset shape:")
print(df.shape)

df.head()

Dataset path:
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Processed_Dataset/diabetic_data_cleaned_stage1.csv

Dataset shape:
(69987, 56)


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,...,diabetesMed,readmitted,readmitted_30,hba1c_group,primary_diagnosis,age_group,discharge_group,race_group,admission_source_group,medical_specialty_group
0,24437208,135,Caucasian,Female,[50-60),2,1,1,8,Cardiology,...,Yes,<30,1,No test was performed,Circulatory,30-60,Home,Caucasian,Physician/clinic referral,Cardiology
1,29758806,378,Caucasian,Female,[50-60),3,1,1,2,Surgery-Neuro,...,No,NO,0,No test was performed,Musculoskeletal,30-60,Home,Caucasian,Physician/clinic referral,Surgery
2,189899286,729,Caucasian,Female,[80-90),1,3,7,4,InternalMedicine,...,Yes,NO,0,Normal result of the test,Injury,>60,Other,Caucasian,Emergency room,Internal Medicine
3,64331490,774,Caucasian,Female,[80-90),1,1,7,3,InternalMedicine,...,Yes,NO,0,"High, medication changed",Other,>60,Home,Caucasian,Emergency room,Internal Medicine
4,14824206,927,AfricanAmerican,Female,[30-40),1,1,7,5,InternalMedicine,...,Yes,NO,0,No test was performed,Genitourinary,30-60,Home,AfricanAmerican,Emergency room,Internal Medicine


## 2. Select the same modelling features used by the other tuned models

For a fair model comparison, this notebook uses exactly the same 18 predictors as the existing tuned models.

In [3]:
target_col = "readmitted_30"

categorical_features = [
    "gender",
    "race_group",
    "age_group",
    "admission_source_group",
    "discharge_group",
    "medical_specialty_group",
    "primary_diagnosis",
    "hba1c_group",
    "max_glu_serum",
    "diabetesMed"
]

numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

model_features = (
    categorical_features
    + numeric_features
)

missing_features = [
    feature
    for feature in model_features
    if feature not in df.columns
]

if missing_features:
    raise ValueError(
        "These modelling features are missing: "
        f"{missing_features}"
    )

X = df[model_features].copy()
y = df[target_col].astype(int).copy()

print("X shape:")
print(X.shape)

print("\nFeatures used:")
print(X.columns.tolist())

print("\nTarget counts:")
print(y.value_counts())

print("\nTarget proportions:")
print(y.value_counts(normalize=True))

X shape:
(69987, 18)

Features used:
['gender', 'race_group', 'age_group', 'admission_source_group', 'discharge_group', 'medical_specialty_group', 'primary_diagnosis', 'hba1c_group', 'max_glu_serum', 'diabetesMed', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Target counts:
readmitted_30
0    63702
1     6285
Name: count, dtype: int64

Target proportions:
readmitted_30
0    0.910198
1    0.089802
Name: proportion, dtype: float64


## 3. CatBoost-specific feature preparation

CatBoost handles categorical variables directly, so this notebook deliberately does **not** use `OneHotEncoder`.

Categorical missing values are replaced by the literal category `"Missing"` and categorical columns are converted to strings. Numeric columns are converted to numeric values; numeric missing values can remain as `NaN` for CatBoost to handle internally.

This preparation does not learn anything from the outcome or from validation/test data, so it does not create leakage.

In [4]:
def prepare_catboost_dataframe(
    X_data,
    categorical_features,
    numeric_features
):
    """Prepare a pandas DataFrame for native CatBoost processing."""

    X_prepared = X_data.copy()

    for feature in categorical_features:
        values = X_prepared[feature].astype("object")
        values = values.where(
            values.notna(),
            "Missing"
        )
        X_prepared[feature] = values.astype(str)

    for feature in numeric_features:
        X_prepared[feature] = pd.to_numeric(
            X_prepared[feature],
            errors="coerce"
        )

    return X_prepared


X = prepare_catboost_dataframe(
    X,
    categorical_features=categorical_features,
    numeric_features=numeric_features
)

print("Categorical dtypes:")
print(X[categorical_features].dtypes)

print("\nNumeric dtypes:")
print(X[numeric_features].dtypes)

print("\nMissing values after preparation:")
print(X.isna().sum())

Categorical dtypes:
gender                     object
race_group                 object
age_group                  object
admission_source_group     object
discharge_group            object
medical_specialty_group    object
primary_diagnosis          object
hba1c_group                object
max_glu_serum              object
diabetesMed                object
dtype: object

Numeric dtypes:
time_in_hospital      int64
num_lab_procedures    int64
num_procedures        int64
num_medications       int64
number_outpatient     int64
number_emergency      int64
number_inpatient      int64
number_diagnoses      int64
dtype: object

Missing values after preparation:
gender                     0
race_group                 0
age_group                  0
admission_source_group     0
discharge_group            0
medical_specialty_group    0
primary_diagnosis          0
hba1c_group                0
max_glu_serum              0
diabetesMed                0
time_in_hospital           0
num_lab_procedure

In [5]:
forbidden_features = {
    "race",
    "age",
    "medical_specialty",
    "diag_1",
    "A1Cresult",
    "admission_type_id",
    "admission_source_id",
    "discharge_disposition_id",
    "readmitted",
    "readmitted_30",
    "encounter_id",
    "patient_nbr"
}

unexpected_features = (
    forbidden_features
    .intersection(X.columns)
)

assert not unexpected_features, (
    "Unexpected or potentially leaking features found: "
    f"{unexpected_features}"
)

assert not X.columns.duplicated().any(), (
    "Duplicate column names were found in X."
)

assert len(X) == len(y), (
    "X and y contain different numbers of rows."
)

assert y.isna().sum() == 0, (
    "The target contains missing values."
)

assert set(y.unique()).issubset({0, 1}), (
    "The target must contain only 0 and 1."
)

for feature in categorical_features:
    assert X[feature].isna().sum() == 0, (
        f"Categorical feature {feature} still contains missing values."
    )

print("Feature and target checks passed.")

Feature and target checks passed.


## 4. Create development, validation, test, and early-stopping sets

The final model-training, validation, and test proportions remain 60%, 20%, and 20% of the full dataset.

An internal early-stopping set is taken from the 64% model-training portion. It is used only to choose the learning rate and number of boosting rounds. It is **not** the validation set used later for threshold selection.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_model_train, X_val, y_model_train, y_val = (
    train_test_split(
        X_train,
        y_train,
        test_size=0.25,
        stratify=y_train,
        random_state=42
    )
)

X_search_train, X_early_stop, y_search_train, y_early_stop = (
    train_test_split(
        X_model_train,
        y_model_train,
        test_size=0.20,
        stratify=y_model_train,
        random_state=42
    )
)

split_summary = pd.DataFrame({
    "split": [
        "CV search training",
        "Internal early stopping",
        "Full model training after selection",
        "Threshold validation",
        "Final test"
    ],
    "rows": [
        len(X_search_train),
        len(X_early_stop),
        len(X_model_train),
        len(X_val),
        len(X_test)
    ],
    "positive_rate": [
        y_search_train.mean(),
        y_early_stop.mean(),
        y_model_train.mean(),
        y_val.mean(),
        y_test.mean()
    ]
})

split_summary

,split,rows,positive_rate
0,CV search training,33592,0.089813
1,Internal early stopping,8399,0.089773
2,Full model training after selection,41991,0.089805
3,Threshold validation,13998,0.089799
4,Final test,13998,0.089799


## 5. Evaluation and threshold-selection helper functions

In [7]:
def evaluate_predictions_from_proba(
    y_true,
    y_proba,
    threshold=0.5,
    model_name="Model"
):
    """Evaluate binary predictions created from probabilities."""

    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    y_pred = (
        y_proba >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    total = tn + fp + fn + tp
    actual_positive = tp + fn
    actual_negative = tn + fp
    predicted_positive = tp + fp
    predicted_negative = tn + fn

    specificity = (
        tn / actual_negative
        if actual_negative > 0
        else np.nan
    )

    false_positive_rate = (
        fp / actual_negative
        if actual_negative > 0
        else np.nan
    )

    false_negative_rate = (
        fn / actual_positive
        if actual_positive > 0
        else np.nan
    )

    predicted_positive_rate = (
        predicted_positive / total
        if total > 0
        else np.nan
    )

    patients_flagged_per_true_readmission = (
        predicted_positive / tp
        if tp > 0
        else np.nan
    )

    return {
        "model": model_name,
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "specificity": specificity,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "f2": fbeta_score(
            y_true,
            y_pred,
            beta=2,
            zero_division=0
        ),
        "auroc": roc_auc_score(
            y_true,
            y_proba
        ),
        "auprc": average_precision_score(
            y_true,
            y_proba
        ),
        "brier_score": brier_score_loss(
            y_true,
            y_proba
        ),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
        "predicted_positive": int(predicted_positive),
        "predicted_negative": int(predicted_negative),
        "predicted_positive_rate": predicted_positive_rate,
        "patients_flagged_per_true_readmission_found": (
            patients_flagged_per_true_readmission
        )
    }

In [8]:
def confusion_matrix_from_proba(
    y_true,
    y_proba,
    threshold=0.5
):
    """Create a labelled confusion matrix from probabilities."""

    y_pred = (
        np.asarray(y_proba) >= threshold
    ).astype(int)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    return pd.DataFrame(
        cm,
        index=[
            "Actual not readmitted",
            "Actual readmitted"
        ],
        columns=[
            "Predicted not readmitted",
            "Predicted readmitted"
        ]
    )

In [9]:
def threshold_sweep(
    y_true,
    y_proba,
    model_name="Model",
    thresholds=None
):
    """Calculate performance across probability thresholds."""

    if thresholds is None:
        thresholds = np.round(
            np.arange(
                0.01,
                0.951,
                0.01
            ),
            2
        )

    results = [
        evaluate_predictions_from_proba(
            y_true=y_true,
            y_proba=y_proba,
            threshold=threshold,
            model_name=model_name
        )
        for threshold in thresholds
    ]

    return pd.DataFrame(results)

In [10]:
def choose_threshold_for_minimum_recall(
    y_true,
    y_proba,
    min_recall=0.80,
    model_name="Model"
):
    """
    Select the threshold with the lowest false-positive rate
    among thresholds that achieve the required recall.
    """

    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    false_positive_rates, recalls, thresholds = roc_curve(
        y_true,
        y_proba,
        drop_intermediate=False
    )

    candidate_table = pd.DataFrame({
        "threshold": thresholds,
        "recall": recalls,
        "false_positive_rate": false_positive_rates,
        "specificity": 1 - false_positive_rates
    })

    candidate_table = candidate_table[
        np.isfinite(
            candidate_table["threshold"]
        )
    ].copy()

    eligible_candidates = candidate_table[
        candidate_table["recall"] >= min_recall
    ].copy()

    if eligible_candidates.empty:
        raise ValueError(
            "No threshold achieved recall >= "
            f"{min_recall:.2f}."
        )

    eligible_candidates = (
        eligible_candidates
        .sort_values(
            by=[
                "false_positive_rate",
                "threshold"
            ],
            ascending=[
                True,
                False
            ]
        )
        .reset_index(drop=True)
    )

    selected_threshold = float(
        eligible_candidates.iloc[0]["threshold"]
    )

    selected_metrics = evaluate_predictions_from_proba(
        y_true=y_true,
        y_proba=y_proba,
        threshold=selected_threshold,
        model_name=model_name
    )

    return (
        selected_threshold,
        selected_metrics,
        eligible_candidates
    )

## 6. Stage 1: tune CatBoost structure, regularisation, categorical combinations, sampling, and class weighting

The first search deliberately keeps `iterations=400` and `learning_rate=0.05` fixed. This lets us compare different tree structures and regularisation settings without mixing the question of *what kind of trees to build* with *how long boosting should continue*.

A manual five-fold cross-validation loop is used so every candidate is ranked by the same scikit-learn **average precision (AUPRC)** measure used in the other optimisation notebooks, while CatBoost still receives the original categorical columns directly.

For a quick code check, change `STRUCTURE_SEARCH_ITERATIONS` from 30 to 5. Restore it to 30 for the proper experiment.

In [11]:
RECALL_TARGET = 0.80
STRUCTURE_SEARCH_ITERATIONS = 30  # Use 5 first for a quick smoke test.
STRUCTURE_SEARCH_TREES = 400
STRUCTURE_SEARCH_LEARNING_RATE = 0.05

cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

negative_count = int((y_search_train == 0).sum())
positive_count = int((y_search_train == 1).sum())
imbalance_ratio = negative_count / positive_count

print(
    f"Required validation recall: "
    f"{RECALL_TARGET:.0%}"
)
print("Cross-validation folds:", cross_validation.n_splits)
print("Structure search candidates:", STRUCTURE_SEARCH_ITERATIONS)
print("Negative to positive ratio:", imbalance_ratio)

Required validation recall: 80%
Cross-validation folds: 5
Structure search candidates: 30
Negative to positive ratio: 10.134239310573417


In [12]:
catboost_structure_param_distributions = {
    "depth": [
        4,
        5,
        6,
        7,
        8,
        9,
        10
    ],

    "l2_leaf_reg": [
        1.0,
        3.0,
        5.0,
        10.0,
        20.0,
        50.0
    ],

    "random_strength": [
        0.0,
        0.5,
        1.0,
        2.0,
        5.0
    ],

    "border_count": [
        64,
        128,
        254
    ],

    "max_ctr_complexity": [
        1,
        2,
        3,
        4
    ],

    "subsample": [
        0.60,
        0.80,
        1.00
    ],

    "scale_pos_weight": [
        1.0,
        2.0,
        4.0,
        6.0,
        float(imbalance_ratio)
    ]
}

sampled_structure_parameters = list(
    ParameterSampler(
        param_distributions=(
            catboost_structure_param_distributions
        ),
        n_iter=STRUCTURE_SEARCH_ITERATIONS,
        random_state=42
    )
)

len(sampled_structure_parameters)

30

In [13]:
def build_catboost_classifier(
    structure_params,
    iterations,
    learning_rate,
    thread_count=-1
):
    """Create a reproducible CatBoost binary classifier."""

    return CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="PRAUC:type=Classic",
        iterations=iterations,
        learning_rate=learning_rate,
        cat_features=categorical_features,
        boosting_type="Plain",
        bootstrap_type="MVS",
        random_seed=42,
        thread_count=thread_count,
        allow_writing_files=False,
        verbose=False,
        **structure_params
    )

In [14]:
catboost_cv_candidate_rows = []
catboost_cv_fold_rows = []

for candidate_number, structure_params in enumerate(
    sampled_structure_parameters,
    start=1
):
    print(
        f"Candidate {candidate_number}/"
        f"{STRUCTURE_SEARCH_ITERATIONS}"
    )

    fold_train_auprc = []
    fold_validation_auprc = []
    fold_validation_auroc = []

    for fold_number, (
        fold_train_indices,
        fold_validation_indices
    ) in enumerate(
        cross_validation.split(
            X_search_train,
            y_search_train
        ),
        start=1
    ):
        X_fold_train = X_search_train.iloc[
            fold_train_indices
        ]
        y_fold_train = y_search_train.iloc[
            fold_train_indices
        ]

        X_fold_validation = X_search_train.iloc[
            fold_validation_indices
        ]
        y_fold_validation = y_search_train.iloc[
            fold_validation_indices
        ]

        fold_model = build_catboost_classifier(
            structure_params=structure_params,
            iterations=STRUCTURE_SEARCH_TREES,
            learning_rate=(
                STRUCTURE_SEARCH_LEARNING_RATE
            ),
            thread_count=-1
        )

        fold_model.fit(
            X_fold_train,
            y_fold_train,
            verbose=False
        )

        train_proba = (
            fold_model
            .predict_proba(X_fold_train)[:, 1]
        )

        validation_proba = (
            fold_model
            .predict_proba(
                X_fold_validation
            )[:, 1]
        )

        train_auprc = average_precision_score(
            y_fold_train,
            train_proba
        )

        validation_auprc = average_precision_score(
            y_fold_validation,
            validation_proba
        )

        validation_auroc = roc_auc_score(
            y_fold_validation,
            validation_proba
        )

        fold_train_auprc.append(
            train_auprc
        )
        fold_validation_auprc.append(
            validation_auprc
        )
        fold_validation_auroc.append(
            validation_auroc
        )

        catboost_cv_fold_rows.append({
            "candidate": candidate_number,
            "fold": fold_number,
            **structure_params,
            "train_auprc": train_auprc,
            "validation_auprc": validation_auprc,
            "validation_auroc": validation_auroc
        })

    mean_train_auprc = float(
        np.mean(fold_train_auprc)
    )

    mean_validation_auprc = float(
        np.mean(fold_validation_auprc)
    )

    catboost_cv_candidate_rows.append({
        "candidate": candidate_number,
        **structure_params,
        "mean_train_auprc": mean_train_auprc,
        "std_train_auprc": float(
            np.std(
                fold_train_auprc,
                ddof=1
            )
        ),
        "mean_validation_auprc": (
            mean_validation_auprc
        ),
        "std_validation_auprc": float(
            np.std(
                fold_validation_auprc,
                ddof=1
            )
        ),
        "mean_validation_auroc": float(
            np.mean(fold_validation_auroc)
        ),
        "std_validation_auroc": float(
            np.std(
                fold_validation_auroc,
                ddof=1
            )
        ),
        "train_validation_auprc_gap": (
            mean_train_auprc
            - mean_validation_auprc
        )
    })

catboost_structure_cv_results = pd.DataFrame(
    catboost_cv_candidate_rows
)

catboost_structure_cv_fold_results = pd.DataFrame(
    catboost_cv_fold_rows
)

catboost_structure_cv_results = (
    catboost_structure_cv_results
    .sort_values(
        by=[
            "mean_validation_auprc",
            "mean_validation_auroc",
            "train_validation_auprc_gap"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

catboost_structure_cv_results[
    "rank_validation_auprc"
] = (
    catboost_structure_cv_results[
        "mean_validation_auprc"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)

catboost_structure_cv_results.to_csv(
    OUTPUT_DIR
    / "catboost_structure_cv_results.csv",
    index=False
)

catboost_structure_cv_fold_results.to_csv(
    OUTPUT_DIR
    / "catboost_structure_cv_fold_results.csv",
    index=False
)

catboost_structure_cv_results.head(20)

Candidate 1/30
Candidate 2/30
Candidate 3/30
Candidate 4/30
Candidate 5/30
Candidate 6/30
Candidate 7/30
Candidate 8/30
Candidate 9/30
Candidate 10/30
Candidate 11/30
Candidate 12/30
Candidate 13/30
Candidate 14/30
Candidate 15/30
Candidate 16/30
Candidate 17/30
Candidate 18/30
Candidate 19/30
Candidate 20/30
Candidate 21/30
Candidate 22/30
Candidate 23/30
Candidate 24/30
Candidate 25/30
Candidate 26/30
Candidate 27/30
Candidate 28/30
Candidate 29/30
Candidate 30/30


,candidate,subsample,scale_pos_weight,random_strength,max_ctr_complexity,l2_leaf_reg,depth,border_count,mean_train_auprc,std_train_auprc,mean_validation_auprc,std_validation_auprc,mean_validation_auroc,std_validation_auroc,train_validation_auprc_gap,rank_validation_auprc
0,2,1.0,2.000000,1.0,4,5.0,4,64,0.192123,0.005967,0.152231,0.012621,0.631930,0.011469,0.039892,1
1,29,0.8,1.000000,1.0,4,50.0,5,64,0.178336,0.005843,0.151778,0.012385,0.631307,0.009809,0.026558,2
2,15,0.8,10.134239,1.0,3,50.0,5,254,0.188848,0.005647,0.151466,0.009048,0.631725,0.009898,0.037382,3
3,8,0.6,2.000000,2.0,2,50.0,5,128,0.184656,0.005167,0.151395,0.013954,0.630689,0.011665,0.033260,4
4,26,0.8,4.000000,5.0,1,20.0,4,64,0.180670,0.004650,0.151203,0.010823,0.629096,0.008738,0.029466,5
5,12,0.8,1.000000,5.0,3,50.0,6,64,0.174749,0.006473,0.151000,0.011884,0.630706,0.010910,0.023750,6
6,22,0.6,6.000000,1.0,3,1.0,4,64,0.196055,0.003895,0.150894,0.010857,0.629168,0.011701,0.045161,7
7,9,1.0,2.000000,1.0,3,50.0,4,64,0.174464,0.005054,0.150768,0.012121,0.630322,0.010589,0.023696,8
8,21,1.0,1.000000,2.0,1,10.0,5,64,0.197686,0.007360,0.150570,0.012013,0.629362,0.009443,0.047116,9
9,10,0.8,2.000000,0.5,3,5.0,4,64,0.197519,0.003848,0.150483,0.011980,0.630133,0.011973,0.047036,10


## 7. Select and inspect the strongest CatBoost structure candidate

In [15]:
best_structure_row = (
    catboost_structure_cv_results
    .iloc[0]
)

selected_structure_params = {
    "depth": int(
        best_structure_row["depth"]
    ),
    "l2_leaf_reg": float(
        best_structure_row["l2_leaf_reg"]
    ),
    "random_strength": float(
        best_structure_row["random_strength"]
    ),
    "border_count": int(
        best_structure_row["border_count"]
    ),
    "max_ctr_complexity": int(
        best_structure_row["max_ctr_complexity"]
    ),
    "subsample": float(
        best_structure_row["subsample"]
    ),
    "scale_pos_weight": float(
        best_structure_row["scale_pos_weight"]
    )
}

print("Selected CatBoost structure parameters:")
for parameter, value in selected_structure_params.items():
    print(f"{parameter}: {value}")

print("\nBest cross-validation AUPRC:")
print(
    best_structure_row[
        "mean_validation_auprc"
    ]
)

print("\nTrain-validation AUPRC gap:")
print(
    best_structure_row[
        "train_validation_auprc_gap"
    ]
)

Selected CatBoost structure parameters:
depth: 4
l2_leaf_reg: 5.0
random_strength: 1.0
border_count: 64
max_ctr_complexity: 4
subsample: 1.0
scale_pos_weight: 2.0

Best cross-validation AUPRC:
0.15223097964631493

Train-validation AUPRC gap:
0.03989196986152729


## 8. Stage 2: choose the learning rate and number of boosting rounds with early stopping

Each learning-rate candidate is allowed to build up to 2,000 trees. Training stops after 75 rounds without improvement in CatBoost's validation PRAUC. We then calculate scikit-learn AUPRC, AUROC, and Brier score on the same internal early-stopping set and select the learning-rate candidate with the highest scikit-learn AUPRC.

The separate threshold-validation set remains untouched.

In [16]:
LEARNING_RATE_CANDIDATES = [
    0.02,
    0.03,
    0.05,
    0.08,
    0.10
]

MAX_BOOSTING_ROUNDS = 2000
EARLY_STOPPING_ROUNDS = 75

learning_rate_results = []
early_stopped_models = {}

for learning_rate in LEARNING_RATE_CANDIDATES:
    candidate_model = build_catboost_classifier(
        structure_params=(
            selected_structure_params
        ),
        iterations=MAX_BOOSTING_ROUNDS,
        learning_rate=learning_rate,
        thread_count=-1
    )

    candidate_model.fit(
        X_search_train,
        y_search_train,
        eval_set=(
            X_early_stop,
            y_early_stop
        ),
        use_best_model=True,
        early_stopping_rounds=(
            EARLY_STOPPING_ROUNDS
        ),
        verbose=False
    )

    early_stop_proba = (
        candidate_model
        .predict_proba(X_early_stop)[:, 1]
    )

    selected_tree_count = int(
        candidate_model.tree_count_
    )

    best_iteration = int(
        candidate_model.get_best_iteration()
    )

    learning_rate_results.append({
        "learning_rate": learning_rate,
        "selected_iterations": (
            selected_tree_count
        ),
        "best_iteration_zero_based": (
            best_iteration
        ),
        "early_stop_auprc": average_precision_score(
            y_early_stop,
            early_stop_proba
        ),
        "early_stop_auroc": roc_auc_score(
            y_early_stop,
            early_stop_proba
        ),
        "early_stop_brier_score": brier_score_loss(
            y_early_stop,
            early_stop_proba
        )
    })

    early_stopped_models[
        learning_rate
    ] = candidate_model

learning_rate_results = (
    pd.DataFrame(learning_rate_results)
    .sort_values(
        by=[
            "early_stop_auprc",
            "early_stop_auroc",
            "selected_iterations"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

learning_rate_results.to_csv(
    OUTPUT_DIR
    / "catboost_learning_rate_early_stopping_results.csv",
    index=False
)

learning_rate_results

,learning_rate,selected_iterations,best_iteration_zero_based,early_stop_auprc,early_stop_auroc,early_stop_brier_score
0,0.10,155,154,0.160171,0.643756,0.085126
1,0.08,74,73,0.158118,0.644968,0.085179
2,0.03,377,376,0.158021,0.644306,0.085181
3,0.05,89,88,0.155789,0.643243,0.085438
4,0.02,25,24,0.154935,0.634797,0.143420


In [17]:
selected_learning_rate = float(
    learning_rate_results.iloc[0][
        "learning_rate"
    ]
)

selected_iterations = int(
    learning_rate_results.iloc[0][
        "selected_iterations"
    ]
)

print("Selected learning rate:")
print(selected_learning_rate)

print("\nSelected number of boosting rounds:")
print(selected_iterations)

Selected learning rate:
0.1

Selected number of boosting rounds:
155


## 9. Refit the selected CatBoost model on the complete model-training set

The internal early-stopping data now returns to the training data. The final selected CatBoost model is fitted on the complete 64% model-training set using the chosen structure, learning rate, and number of trees. The threshold-validation and final-test sets are still untouched.

In [18]:
final_catboost_params = deepcopy(
    selected_structure_params
)

best_catboost_model = build_catboost_classifier(
    structure_params=final_catboost_params,
    iterations=selected_iterations,
    learning_rate=selected_learning_rate,
    thread_count=-1
)

best_catboost_model.fit(
    X_model_train,
    y_model_train,
    verbose=False
)

best_catboost_model

CatBoostClassifier(allow_writing_files=False, boosting_type='Plain', bootstrap_type='MVS', border_count=64, cat_features=['gender', 'race_group', 'age_group', 'admission_source_group', 'discharge_group', 'medical_specialty_group', 'primary_diagnosis', 'hba1c_group', 'max_glu_serum', 'diabetesMed'], depth=4, eval_metric='PRAUC:type=Classic', iterations=155, l2_leaf_reg=5.0, learning_rate=0.1, loss_function='Logloss', max_ctr_complexity=4, random_seed=42, random_strength=1.0, scale_pos_weight=2.0, subsample=1.0, verbose=False)

In [19]:
catboost_structure_summary = pd.Series({
    "number_of_boosting_rounds": int(
        best_catboost_model.tree_count_
    ),
    "learning_rate": float(
        selected_learning_rate
    ),
    "depth": selected_structure_params[
        "depth"
    ],
    "l2_leaf_reg": selected_structure_params[
        "l2_leaf_reg"
    ],
    "random_strength": selected_structure_params[
        "random_strength"
    ],
    "border_count": selected_structure_params[
        "border_count"
    ],
    "max_ctr_complexity": selected_structure_params[
        "max_ctr_complexity"
    ],
    "subsample": selected_structure_params[
        "subsample"
    ],
    "scale_pos_weight": selected_structure_params[
        "scale_pos_weight"
    ],
    "boosting_type": "Plain",
    "bootstrap_type": "MVS"
})

catboost_structure_summary.to_csv(
    OUTPUT_DIR
    / "catboost_selected_structure_summary.csv",
    header=["value"]
)

catboost_structure_summary

number_of_boosting_rounds      155
learning_rate                  0.1
depth                            4
l2_leaf_reg                    5.0
random_strength                1.0
border_count                    64
max_ctr_complexity               4
subsample                      1.0
scale_pos_weight               2.0
boosting_type                Plain
bootstrap_type                 MVS
dtype: object

## 10. Evaluate the selected model on the validation set at the default threshold

In [20]:
y_val_proba_catboost = (
    best_catboost_model
    .predict_proba(X_val)[:, 1]
)

print(
    "Minimum validation probability:",
    y_val_proba_catboost.min()
)

print(
    "Maximum validation probability:",
    y_val_proba_catboost.max()
)

print(
    "Number of unique validation probabilities:",
    np.unique(y_val_proba_catboost).size
)

Minimum validation probability: 0.05418168008976404
Maximum validation probability: 0.6727816171955074
Number of unique validation probabilities: 13967


In [21]:
catboost_probability_summary = (
    pd.DataFrame({
        "actual_class": np.asarray(y_val),
        "predicted_readmission_probability": (
            y_val_proba_catboost
        )
    })
    .groupby("actual_class")
    ["predicted_readmission_probability"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

catboost_probability_summary.to_csv(
    OUTPUT_DIR
    / "catboost_validation_probability_summary.csv"
)

catboost_probability_summary

,count,mean,std,min,10%,25%,50%,75%,90%,max
actual_class,,,,,,,,,,
0,12741.0,0.159506,0.061896,0.054182,0.095299,0.112614,0.142851,0.200844,0.232612,0.672782
1,1257.0,0.187993,0.073128,0.067495,0.108440,0.132190,0.184089,0.224367,0.266562,0.532976


In [22]:
catboost_val_default_results = (
    evaluate_predictions_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_catboost,
        threshold=0.5,
        model_name=(
            "CatBoost validation default"
        )
    )
)

pd.Series(
    catboost_val_default_results
)

model                                          CatBoost validation default
threshold                                                              0.5
accuracy                                                          0.909559
precision                                                         0.263158
recall                                                            0.003978
specificity                                                       0.998901
false_positive_rate                                               0.001099
false_negative_rate                                               0.996022
f1                                                                0.007837
f2                                                                0.004953
auroc                                                             0.624723
auprc                                                             0.138084
brier_score                                                       0.086333
true_negative            

## 11. Select the validation threshold that reaches at least 80% recall

Among all validation thresholds that achieve the recall target, the rule selects the threshold with the lowest false-positive rate. This is the same operating-point rule used for the other tuned models.

In [23]:
(
    catboost_selected_threshold,
    catboost_val_selected_results,
    catboost_eligible_thresholds
) = choose_threshold_for_minimum_recall(
    y_true=y_val,
    y_proba=y_val_proba_catboost,
    min_recall=RECALL_TARGET,
    model_name=(
        "CatBoost validation selected"
    )
)

print("Selected CatBoost threshold:")
print(catboost_selected_threshold)

pd.DataFrame([
    catboost_val_default_results,
    catboost_val_selected_results
])

Selected CatBoost threshold:
0.12473816909208556


,model,threshold,accuracy,precision,recall,specificity,false_positive_rate,false_negative_rate,f1,f2,...,auprc,brier_score,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,CatBoost validation default,0.500000,0.909559,0.263158,0.003978,0.998901,0.001099,0.996022,0.007837,0.004953,...,0.138084,0.086333,12727,14,1252,5,19,13979,0.001357,3.800000
1,CatBoost validation selected,0.124738,0.402486,0.110319,0.800318,0.363237,0.636763,0.199682,0.193909,0.355552,...,0.138084,0.086333,4628,8113,251,1006,9119,4879,0.651450,9.064612


In [24]:
catboost_eligible_thresholds.to_csv(
    OUTPUT_DIR
    / "catboost_eligible_validation_thresholds.csv",
    index=False
)

catboost_eligible_thresholds.head(20)

,threshold,recall,false_positive_rate,specificity
0,0.124738,0.800318,0.636763,0.363237
1,0.124726,0.800318,0.636842,0.363158
2,0.124725,0.800318,0.636920,0.363080
3,0.124711,0.800318,0.636999,0.363001
4,0.124703,0.800318,0.637077,0.362923
5,0.124700,0.800318,0.637156,0.362844
6,0.124694,0.800318,0.637234,0.362766
7,0.124693,0.800318,0.637313,0.362687
8,0.124690,0.800318,0.637391,0.362609
9,0.124679,0.800318,0.637470,0.362530


In [25]:
catboost_threshold_sweep = threshold_sweep(
    y_true=y_val,
    y_proba=y_val_proba_catboost,
    model_name="CatBoost validation"
)

catboost_threshold_sweep.to_csv(
    OUTPUT_DIR
    / "catboost_validation_threshold_sweep.csv",
    index=False
)

catboost_threshold_sweep[
    [
        "threshold",
        "recall",
        "precision",
        "specificity",
        "false_positive_rate",
        "f1",
        "f2",
        "true_positive",
        "false_positive",
        "false_negative",
        "predicted_positive_rate"
    ]
]

,threshold,recall,precision,specificity,false_positive_rate,f1,f2,true_positive,false_positive,false_negative,predicted_positive_rate
0,0.01,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
1,0.02,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
2,0.03,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
3,0.04,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
4,0.05,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
90,0.91,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0
91,0.92,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0
92,0.93,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0
93,0.94,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0


In [26]:
comparison_columns = [
    "model",
    "threshold",
    "auprc",
    "auroc",
    "brier_score",
    "accuracy",
    "recall",
    "precision",
    "specificity",
    "false_positive_rate",
    "false_negative_rate",
    "f1",
    "f2",
    "true_positive",
    "true_negative",
    "false_positive",
    "false_negative",
    "predicted_positive_rate",
    "patients_flagged_per_true_readmission_found"
]

catboost_validation_comparison = pd.DataFrame([
    catboost_val_default_results,
    catboost_val_selected_results
])

catboost_validation_comparison = (
    catboost_validation_comparison[
        comparison_columns
    ]
)

catboost_validation_comparison.to_csv(
    OUTPUT_DIR
    / "catboost_validation_results.csv",
    index=False
)

catboost_validation_comparison

,model,threshold,auprc,auroc,brier_score,accuracy,recall,precision,specificity,false_positive_rate,false_negative_rate,f1,f2,true_positive,true_negative,false_positive,false_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,CatBoost validation default,0.500000,0.138084,0.624723,0.086333,0.909559,0.003978,0.263158,0.998901,0.001099,0.996022,0.007837,0.004953,5,12727,14,1252,0.001357,3.800000
1,CatBoost validation selected,0.124738,0.138084,0.624723,0.086333,0.402486,0.800318,0.110319,0.363237,0.636763,0.199682,0.193909,0.355552,1006,4628,8113,251,0.651450,9.064612


In [27]:
catboost_val_selected_cm = (
    confusion_matrix_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_catboost,
        threshold=(
            catboost_selected_threshold
        )
    )
)

catboost_val_selected_cm.to_csv(
    OUTPUT_DIR
    / "catboost_validation_confusion_matrix.csv"
)

catboost_val_selected_cm

,Predicted not readmitted,Predicted readmitted
Actual not readmitted,4628,8113
Actual readmitted,251,1006


In [28]:
y_val_pred_catboost = (
    y_val_proba_catboost
    >= catboost_selected_threshold
).astype(int)

print(
    classification_report(
        y_val,
        y_val_pred_catboost,
        target_names=[
            "Not readmitted",
            "Readmitted"
        ],
        zero_division=0
    )
)

                precision    recall  f1-score   support

Not readmitted       0.95      0.36      0.53     12741
    Readmitted       0.11      0.80      0.19      1257

      accuracy                           0.40     13998
     macro avg       0.53      0.58      0.36     13998
  weighted avg       0.87      0.40      0.50     13998



## 12. Final evaluation on the untouched test set

In [29]:
y_test_proba_catboost = (
    best_catboost_model
    .predict_proba(X_test)[:, 1]
)

final_catboost_test_results = (
    evaluate_predictions_from_proba(
        y_true=y_test,
        y_proba=y_test_proba_catboost,
        threshold=(
            catboost_selected_threshold
        ),
        model_name="Final CatBoost test"
    )
)

final_catboost_test_results_df = pd.DataFrame([
    final_catboost_test_results
])

final_catboost_test_results_df

,model,threshold,accuracy,precision,recall,specificity,false_positive_rate,false_negative_rate,f1,f2,...,auprc,brier_score,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Final CatBoost test,0.124738,0.409701,0.113355,0.817025,0.369516,0.630484,0.182975,0.199089,0.364495,...,0.15158,0.0857,4708,8033,230,1027,9060,4938,0.647235,8.821811


In [30]:
final_catboost_test_cm = (
    confusion_matrix_from_proba(
        y_true=y_test,
        y_proba=y_test_proba_catboost,
        threshold=(
            catboost_selected_threshold
        )
    )
)

final_catboost_test_cm

,Predicted not readmitted,Predicted readmitted
Actual not readmitted,4708,8033
Actual readmitted,230,1027


In [31]:
y_test_pred_catboost = (
    y_test_proba_catboost
    >= catboost_selected_threshold
).astype(int)

print(
    classification_report(
        y_test,
        y_test_pred_catboost,
        target_names=[
            "Not readmitted",
            "Readmitted"
        ],
        zero_division=0
    )
)

                precision    recall  f1-score   support

Not readmitted       0.95      0.37      0.53     12741
    Readmitted       0.11      0.82      0.20      1257

      accuracy                           0.41     13998
     macro avg       0.53      0.59      0.37     13998
  weighted avg       0.88      0.41      0.50     13998



In [32]:
catboost_validation_test_comparison = (
    pd.DataFrame([
        catboost_val_selected_results,
        final_catboost_test_results
    ])
)

catboost_validation_test_comparison[
    [
        "model",
        "threshold",
        "auprc",
        "auroc",
        "brier_score",
        "recall",
        "precision",
        "specificity",
        "false_positive_rate",
        "f2",
        "predicted_positive_rate",
        "patients_flagged_per_true_readmission_found"
    ]
]

,model,threshold,auprc,auroc,brier_score,recall,precision,specificity,false_positive_rate,f2,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,CatBoost validation selected,0.124738,0.138084,0.624723,0.086333,0.800318,0.110319,0.363237,0.636763,0.355552,0.651450,9.064612
1,Final CatBoost test,0.124738,0.151580,0.642073,0.085700,0.817025,0.113355,0.369516,0.630484,0.364495,0.647235,8.821811


In [33]:
final_catboost_test_results_df.to_csv(
    OUTPUT_DIR
    / "final_catboost_test_metrics.csv",
    index=False
)

final_catboost_test_cm.to_csv(
    OUTPUT_DIR
    / "final_catboost_test_confusion_matrix.csv"
)

catboost_validation_test_comparison.to_csv(
    OUTPUT_DIR
    / "catboost_validation_test_comparison.csv",
    index=False
)

print("Final CatBoost results saved to:")
print(OUTPUT_DIR)

Final CatBoost results saved to:
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Model_Results/catboost_optimisation


## 13. CatBoost feature importance

CatBoost reports importance on the **original input variables**, so there is no need to add one-hot category importances back together. For binary classification, the standard feature importance is based on how much predictions change when a feature is used.

This is useful for an initial inspection, but it does not show whether a feature raises or lowers risk and it is not causal. SHAP or permutation importance can be used later for stronger interpretation.

In [34]:
catboost_feature_importance = (
    pd.DataFrame({
        "feature": model_features,
        "importance": (
            best_catboost_model
            .get_feature_importance(
                type="PredictionValuesChange"
            )
        )
    })
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

catboost_feature_importance.to_csv(
    OUTPUT_DIR
    / "catboost_feature_importance.csv",
    index=False
)

catboost_feature_importance

,feature,importance
0,discharge_group,25.549650
1,number_inpatient,21.404389
2,age_group,8.507393
3,time_in_hospital,7.589540
4,primary_diagnosis,5.557950
5,num_medications,4.322944
6,number_diagnoses,3.888444
7,num_lab_procedures,3.710784
8,admission_source_group,3.461767
9,diabetesMed,3.001016


## 14. Save the selected settings, final model, and experiment summary

In [35]:
best_structure_parameters_table = pd.DataFrame(
    [
        {
            "parameter": parameter,
            "selected_value": str(value)
        }
        for parameter, value
        in selected_structure_params.items()
    ]
)

best_structure_parameters_table.to_csv(
    OUTPUT_DIR
    / "catboost_best_structure_parameters.csv",
    index=False
)

selected_boosting_schedule_table = pd.DataFrame([
    {
        "selected_learning_rate": (
            selected_learning_rate
        ),
        "selected_iterations": (
            selected_iterations
        ),
        "early_stopping_rounds": (
            EARLY_STOPPING_ROUNDS
        ),
        "maximum_boosting_rounds_tested": (
            MAX_BOOSTING_ROUNDS
        )
    }
])

selected_boosting_schedule_table.to_csv(
    OUTPUT_DIR
    / "catboost_selected_boosting_schedule.csv",
    index=False
)

selected_threshold_table = pd.DataFrame([
    {
        "recall_target": RECALL_TARGET,
        "selected_validation_threshold": (
            catboost_selected_threshold
        )
    }
])

selected_threshold_table.to_csv(
    OUTPUT_DIR
    / "catboost_selected_threshold.csv",
    index=False
)

best_catboost_model.save_model(
    str(
        OUTPUT_DIR
        / "catboost_final_model.cbm"
    )
)

experiment_summary = pd.Series({
    "structure_search_candidates": (
        STRUCTURE_SEARCH_ITERATIONS
    ),
    "structure_search_cv_folds": (
        cross_validation.n_splits
    ),
    "structure_search_cross_validation_auprc": (
        best_structure_row[
            "mean_validation_auprc"
        ]
    ),
    "structure_search_train_validation_auprc_gap": (
        best_structure_row[
            "train_validation_auprc_gap"
        ]
    ),
    "selected_learning_rate": (
        selected_learning_rate
    ),
    "selected_iterations": (
        selected_iterations
    ),
    "selected_validation_threshold": (
        catboost_selected_threshold
    ),
    "validation_recall": (
        catboost_val_selected_results[
            "recall"
        ]
    ),
    "validation_precision": (
        catboost_val_selected_results[
            "precision"
        ]
    ),
    "validation_false_positive_rate": (
        catboost_val_selected_results[
            "false_positive_rate"
        ]
    ),
    "test_auprc": (
        final_catboost_test_results[
            "auprc"
        ]
    ),
    "test_auroc": (
        final_catboost_test_results[
            "auroc"
        ]
    ),
    "test_brier_score": (
        final_catboost_test_results[
            "brier_score"
        ]
    ),
    "test_accuracy": (
        final_catboost_test_results[
            "accuracy"
        ]
    ),
    "test_recall": (
        final_catboost_test_results[
            "recall"
        ]
    ),
    "test_precision": (
        final_catboost_test_results[
            "precision"
        ]
    ),
    "test_specificity": (
        final_catboost_test_results[
            "specificity"
        ]
    ),
    "test_false_positive_rate": (
        final_catboost_test_results[
            "false_positive_rate"
        ]
    ),
    "test_f2": (
        final_catboost_test_results[
            "f2"
        ]
    ),
    "test_predicted_positive_rate": (
        final_catboost_test_results[
            "predicted_positive_rate"
        ]
    ),
    "test_patients_flagged_per_true_readmission": (
        final_catboost_test_results[
            "patients_flagged_per_true_readmission_found"
        ]
    )
})

experiment_summary.to_csv(
    OUTPUT_DIR
    / "catboost_experiment_summary.csv",
    header=["value"]
)

experiment_summary

structure_search_candidates                     30.000000
structure_search_cv_folds                        5.000000
structure_search_cross_validation_auprc          0.152231
structure_search_train_validation_auprc_gap      0.039892
selected_learning_rate                           0.100000
selected_iterations                            155.000000
selected_validation_threshold                    0.124738
validation_recall                                0.800318
validation_precision                             0.110319
validation_false_positive_rate                   0.636763
test_auprc                                       0.151580
test_auroc                                       0.642073
test_brier_score                                 0.085700
test_accuracy                                    0.409701
test_recall                                      0.817025
test_precision                                   0.113355
test_specificity                                 0.369516
test_false_pos

## 15. Main outputs for later model comparison

The most important files are:

- `final_catboost_test_metrics.csv`
- `final_catboost_test_confusion_matrix.csv`
- `catboost_validation_test_comparison.csv`
- `catboost_experiment_summary.csv`
- `catboost_feature_importance.csv`

Additional tuning evidence is saved in:

- `catboost_structure_cv_results.csv`
- `catboost_structure_cv_fold_results.csv`
- `catboost_learning_rate_early_stopping_results.csv`
- `catboost_best_structure_parameters.csv`
- `catboost_selected_boosting_schedule.csv`
- `catboost_selected_threshold.csv`
- `catboost_final_model.cbm`

For the final comparison, place most weight on test AUPRC, AUROC, recall, false-positive rate, precision, Brier score, predicted-positive rate, and patients flagged per true readmission found. Accuracy alone is not suitable for this imbalanced outcome.